In [ ]:
# %pip install firthmodels

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.ticker import PercentFormatter
import matplotlib as mpl
import textwrap
import gc

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

# Loading and join

In [ ]:
# Load nvf lookup
csew_vf = pd.read_csv(
    f_root / "data/csew/merged/vf_lookup.csv",
    dtype=str,
    index_col="global_person_id",
    na_values=["<NA>", "NaN"])

# Load nvf lookup
csew_nvf = pd.read_csv(
    f_root / "data/csew/merged/nvf_lookup.csv",
    dtype=str,
    index_col="global_person_id",
    na_values=["<NA>", "NaN"])

csew_vf_nvf = (
    csew_nvf
    .assign(received_vf=lambda x: x.index.isin(csew_vf.index)).join(
        csew_vf,
        how="left",
        rsuffix="_vf",
        validate="one_to_many"
    ))
csew_vf_nvf["received_vf"] = (csew_vf_nvf["received_vf"].astype("boolean"))
csew_vf_nvf.drop(
    columns=["Unnamed: 0", "Unnamed: 0_vf", "year", "wave_vf"],
    inplace=True,
    errors="ignore")

vawg_cols = ["vawg_official_codes", "vawg_sex_force_threats","vawg_domestic", "vawg_harasm"]
csew_vf_nvf["vawg"] = (
    csew_vf_nvf[vawg_cols]
    .eq("True")
    .any(axis=1)
    .astype("boolean"))

csew_vf_nvf_females = csew_vf_nvf[csew_vf_nvf['sex'] == "2"].copy()

aggregation = {
    col: "first"
    for col in csew_vf_nvf_females.columns
    if col != "vawg"}
aggregation["vawg"] = (lambda x: int(pd.to_numeric(x, errors="coerce").eq(1).any()))
vawg_person_level = (csew_vf_nvf_females.groupby(level=0).agg(aggregation))

del csew_vf, csew_nvf, csew_vf_nvf, csew_vf_nvf_females
gc.collect()

vawg_person_level

# Strata lookup table generation

In [ ]:
# -----------------------------
# Analysis parameters
# -----------------------------

post_strat_vars = ["agelong","remploya","gor"]
weight_col, outcome_col = "ind_weight","vawg"
cell_col = "_".join(post_strat_vars)

n_boot, confidence_level, random_seed = 2000, .95, 12345
lower_q, upper_q = (1-confidence_level)/2, 1-(1-confidence_level)/2

# -----------------------------
# Prepare analysis data
# -----------------------------
df = vawg_person_level[post_strat_vars+[weight_col,outcome_col]].dropna().copy()
df[weight_col] = pd.to_numeric(df[weight_col],errors="coerce")
df = df.loc[np.isfinite(df[weight_col]) & df[weight_col].gt(0)
            & df[outcome_col].isin([0,1,False,True])].copy()

df[outcome_col] = df[outcome_col].astype(int)
df[cell_col] = df[post_strat_vars].astype("string").apply(lambda x:x.str.strip()).agg("_".join,axis=1)
df["_weighted_victim"] = df[weight_col]*df[outcome_col]

# Original person-level table no longer needed for this step
gc.collect()

# -----------------------------
# Point estimates
# -----------------------------
victim_lookup = (
    df.groupby(post_strat_vars+[cell_col],observed=True)
      .agg(weighted_victims=("_weighted_victim","sum"),
           total_weight=(weight_col,"sum"),
           unweighted_n=(outcome_col,"size"),
           unweighted_victims=(outcome_col,"sum"))
      .assign(share_pop_victim=lambda x:x.weighted_victims/x.total_weight,
              percent_pop_victim=lambda x:100*x.share_pop_victim)
      .reset_index()
)

all_cells = victim_lookup[cell_col].tolist()
cell_index = {c:i for i,c in enumerate(all_cells)}

# -----------------------------
# Respondent bootstrap
# -----------------------------
rng = np.random.default_rng(random_seed)
boot = np.full((n_boot,len(all_cells)),np.nan,dtype=np.float32)

for b in range(n_boot):
    idx = rng.integers(0,len(df),len(df))
    s = df.iloc[idx].groupby(cell_col,observed=True).agg(
        num=("_weighted_victim","sum"),
        den=(weight_col,"sum")
    )
    est = s["num"]/s["den"]
    for cell,value in est.items():
        boot[b,cell_index[cell]] = value

# -----------------------------
# Bootstrap intervals
# -----------------------------
bootstrap_summary = pd.DataFrame({
    cell_col: all_cells,
    "bootstrap_mean": np.nanmean(boot,axis=0),
    "bootstrap_se": np.nanstd(boot,axis=0,ddof=1),
    "ci_lower": np.nanquantile(boot,lower_q,axis=0),
    "ci_upper": np.nanquantile(boot,upper_q,axis=0),
    "valid_bootstrap_draws": np.sum(~np.isnan(boot),axis=0)
})

del boot
gc.collect()

# -----------------------------
# Final prevalence lookup
# -----------------------------
victim_lookup = (
    victim_lookup.merge(bootstrap_summary,on=cell_col,how="left",validate="one_to_one")
    .assign(percent_bootstrap_se=lambda x:100*x.bootstrap_se,
            percent_ci_lower=lambda x:100*x.ci_lower,
            percent_ci_upper=lambda x:100*x.ci_upper)
    .sort_values(post_strat_vars)
    .reset_index(drop=True)
)

# -----------------------------
# Analysis parameter table
# -----------------------------
bootstrap_parameters = pd.DataFrame({
    "parameter":["Post-stratification variables","Cell variable","Outcome","Survey weight",
                 "Analysis respondents","Observed post-stratification cells",
                 "Bootstrap repetitions","Confidence level","Lower interval percentile",
                 "Upper interval percentile","Random seed","Bootstrap unit","Interval method"],
    "value":[", ".join(post_strat_vars),cell_col,outcome_col,weight_col,
             f"{len(df):,}",f"{len(all_cells):,}",f"{n_boot:,}",f"{confidence_level:.0%}",
             f"{100*lower_q:.1f}%",f"{100*upper_q:.1f}%",random_seed,
             "Person/respondent","Percentile bootstrap"]
})

del bootstrap_summary
gc.collect()

display(victim_lookup)
display(bootstrap_parameters)

# Export

In [ ]:
victim_lookup.columns

In [ ]:
victim_lookup.to_csv(f_root / "data/csew/exp_for_inla_stage/perc_pop_victim.csv")

# Visualisation

In [ ]:
plt.rcParams["font.family"] = "Roboto Slab"

post_strat_vars = ["agelong","remploya"]
weight_col, vf_col, vawg_col = "ind_weight","received_vf","vawg"
labels = {
    "age_16_24":"16–24","age_25_34":"25–34","age_35_64":"35–64","age_65_plus":"65+",
    "employed":"Employed","unemployed_or_economically_inactive":"Unemployed or\neconomically inactive",
    "All respondents":"All respondents"
}
bar_cols = ["#a3cef1","#ffd670","#ef233c"]

df = vawg_person_level.copy()
df[weight_col] = pd.to_numeric(df[weight_col],errors="coerce")
df[[vf_col,vawg_col]] = df[[vf_col,vawg_col]].astype("boolean")
df = df.loc[np.isfinite(df[weight_col]) & df[weight_col].gt(0)]

rows = []
for var in post_strat_vars:
    t = df.dropna(subset=[var,vf_col]).copy()
    t["_no_vf"] = t[vf_col].eq(False)
    t["_non_vawg"] = t[vf_col].eq(True) & t[vawg_col].eq(False)
    t["_vawg"] = t[vf_col].eq(True) & t[vawg_col].eq(True)
    t = t.loc[t[["_no_vf","_non_vawg","_vawg"]].any(axis=1)]

    for c in ["_no_vf","_non_vawg","_vawg"]:
        t[c+"_w"] = t[c].astype(int)*t[weight_col]

    s = (t.groupby(var,observed=True)
         .agg(total=(weight_col,"sum"),no_vf_w=("_no_vf_w","sum"),
              non_vawg_w=("_non_vawg_w","sum"),vawg_w=("_vawg_w","sum"),
              unweighted_n=(weight_col,"size"))
         .assign(no_vf=lambda x:x.no_vf_w/x.total,
                 non_vawg=lambda x:x.non_vawg_w/x.total,
                 vawg=lambda x:x.vawg_w/x.total)
         .reset_index(names="category").assign(variable=var))
    rows.append(s)

overall = df.dropna(subset=[vf_col]).copy()
overall["_no_vf"] = overall[vf_col].eq(False)
overall["_non_vawg"] = overall[vf_col].eq(True) & overall[vawg_col].eq(False)
overall["_vawg"] = overall[vf_col].eq(True) & overall[vawg_col].eq(True)
overall = overall.loc[overall[["_no_vf","_non_vawg","_vawg"]].any(axis=1)]
w = overall[weight_col]

overall_row = pd.DataFrame([{
    "variable":"Overall","category":"All respondents",
    "no_vf":w[overall["_no_vf"]].sum()/w.sum(),
    "non_vawg":w[overall["_non_vawg"]].sum()/w.sum(),
    "vawg":w[overall["_vawg"]].sum()/w.sum(),
    "unweighted_n":len(overall)
}])

plot_df = pd.concat(rows+[overall_row],ignore_index=True)
plot_df["variable"] = pd.Categorical(
    plot_df["variable"],categories=post_strat_vars+["Overall"],ordered=True)
plot_df = plot_df.sort_values(["variable","category"]).reset_index(drop=True)

x = np.arange(len(plot_df))
fig,ax = plt.subplots(figsize=(12,3))
bottom = np.zeros(len(plot_df))

for (label,col),colour in zip({
    "No victim form":"no_vf",
    "Victimised: non-VAWG":"non_vawg",
    "Victimised: VAWG":"vawg"
}.items(),bar_cols):
    values = plot_df[col].to_numpy()
    bars = ax.bar(x,values,bottom=bottom,label=label,color=colour)
    for i,(bar,v) in enumerate(zip(bars,values)):
        if v >= .005:
            ax.text(bar.get_x()+bar.get_width()/2,bottom[i]+v/2,f"{v:.1%}",
                    ha="center",va="center",fontsize=9,
                    path_effects=[pe.Stroke(linewidth=2.5,foreground="white"),pe.Normal()])
    bottom += values

ax.set(xticks=x,
       xticklabels=[textwrap.fill(labels.get(str(v),str(v)),12) for v in plot_df["category"]],
       ylim=(0,1),ylabel="Weighted share of category population")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.tick_params(axis="x",labelrotation=0)

for pos in plot_df.groupby("variable",observed=True).size().cumsum().iloc[:-1]:
    ax.axvline(pos-.5,linewidth=.6)

ax.legend(title="Respondent classification",bbox_to_anchor=(1.02,.5),
          loc="center left",frameon=False)
ax.spines[["top","right"]].set_visible(False)

fig.subplots_adjust(right=.78,bottom=.16)
plt.show()

In [ ]:
fig.savefig(
    f_root / "figures/vawg_victims_in_sample_final.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)